# Apollo: Restore a track → better (perceived) bit rate

**Input:** One audio track (e.g. MP3 or WAV, possibly compressed).  
**Output:** Restored WAV with higher perceived quality (Apollo removes codec artifacts and restores mid/high frequencies).

Apollo is a band-sequence model for **music restoration** in compressed audio ([paper](https://arxiv.org/abs/2409.08514), [Hugging Face](https://huggingface.co/JusperLee/Apollo)). It expects **44.1 kHz** mono/stereo; we resample if needed.

**GPU:** Use **Python (Apollo)** (`make apollo-kernel`) or **Python (rocm-pytorch GPU)**; both use ROCm PyTorch and run on AMD GPU when available. If the kernel crashes (e.g. RX 6600 XT), set **FORCE_CPU = True** or add `export HSA_OVERRIDE_GFX_VERSION=10.3.0` in `~/.bashrc` and restart Cursor.

In [11]:
# Run this once (ROCm kernel). For MP3 input you need ffmpeg: sudo apt install ffmpeg
# Numba (used by librosa) needs NumPy < 2.4 — pin it first
%pip install librosa soundfile huggingface_hub omegaconf pydub -q

Note: you may need to restart the kernel to use updated packages.


In [12]:
# Setup: add Apollo to path (works whether cwd is Applications/ or repo root)
import sys
import os

cwd = os.getcwd()
apollo_dir = os.path.abspath(os.path.join(cwd, "Apollo"))
if not os.path.isdir(apollo_dir):
    apollo_dir = os.path.abspath(os.path.join(cwd, "Applications", "Apollo"))
if os.path.isdir(apollo_dir) and apollo_dir not in sys.path:
    sys.path.insert(0, apollo_dir)
app_dir = os.path.dirname(apollo_dir)  # Applications/

In [13]:
# Load Apollo model (downloads from Hugging Face on first run)
# For GPU: use kernel "Python (rocm-pytorch GPU)". If it crashes, set FORCE_CPU = True.
FORCE_CPU = False  # True = CPU only (stable); False = use GPU when available
import sys, os
# look2hear is in repo Applications/Apollo/look2hear (not on PyPI). Find Apollo from cwd or parents.
_cwd = os.path.abspath(os.getcwd())
_apollo = None
for _base in [_cwd] + [os.path.dirname(_cwd)] * 4:  # cwd and up to 4 parents
    if not _base:
        break
    for _sub in ("Apollo", os.path.join("Applications", "Apollo")):
        _d = os.path.join(_base, _sub)
        if os.path.isdir(_d) and os.path.isdir(os.path.join(_d, "look2hear", "models")):
            _apollo = os.path.abspath(_d)
            break
    if _apollo:
        break
if _apollo and _apollo not in sys.path:
    sys.path.insert(0, _apollo)
if not _apollo:
    raise RuntimeError("Apollo dir not found (looked from cwd=%s and parents)." % _cwd)
import torch
from huggingface_hub import hf_hub_download

device = torch.device("cpu" if FORCE_CPU else ("cuda" if torch.cuda.is_available() else "cpu"))
print(f"Using device: {device}")
if device.type == "cuda":
    print("(GPU active. Use FORCE_CPU = True if the kernel crashes.)")
elif not FORCE_CPU:
    print("Tip: select kernel 'Python (rocm-pytorch GPU)' for AMD GPU.")

# Download checkpoint and load via Apollo's from_pretrain
model_path = hf_hub_download(repo_id="JusperLee/Apollo", filename="pytorch_model.bin")
import look2hear.models
model = look2hear.models.BaseModel.from_pretrain(
    model_path, sr=44100, win=20, feature_dim=256, layer=6
)
model = model.to(device)
model.eval()

TARGET_SR = 44100

Using device: cuda
(GPU active. Use FORCE_CPU = True if the kernel crashes.)
[5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 47] 80


In [14]:
import numpy as np
import librosa
import soundfile as sf
from pydub import AudioSegment

def load_audio(path, target_sr=44100):
    """Load WAV/MP3 etc. Return [1, C, T] on device at target_sr.
    - WAV: preserve channel count (mono or stereo).
    - MP3/others: currently converted to mono via pydub+ffmpeg.
    """
    path = os.path.abspath(path)
    ext = os.path.splitext(path)[1].lower()
    if ext in (".mp3", ".m4a", ".aac", ".ogg", ".flac"):
        # pydub + ffmpeg (avoids libsndfile which doesn't support MP3 well)
        seg = AudioSegment.from_file(path)
        seg = seg.set_channels(1)  # keep mono for now to limit VRAM
        sr = seg.frame_rate
        samples = np.array(seg.get_array_of_samples(), dtype=np.float32) / (2**15)
        if sr != target_sr:
            samples = librosa.resample(samples.astype(np.float64), orig_sr=sr, target_sr=target_sr)
        y = samples.astype(np.float32)[None, :]  # [1, T]
    else:
        # WAV and friends: preserve channels
        y, _ = librosa.load(path, sr=target_sr, mono=False)
        if y.ndim == 1:
            y = y[None, :]  # [1, T]
        # librosa returns [C, T]
    waveform = torch.from_numpy(y).float().unsqueeze(0).to(device)  # [1, C, T]
    return waveform

def save_audio(path, waveform, sr=44100):
    """Save tensor [1, C, T] or [1, T] to 16-bit WAV with soundfile (preserve channels)."""
    w = waveform.squeeze(0).cpu().numpy()  # [C, T] or [T]
    if w.ndim == 1:
        w = w[None, :]
    w = w.T  # [T, C]
    sf.write(path, w, sr, subtype="PCM_16")


def restore_in_chunks(model, audio, chunk_seconds=3.0, overlap_hop_seconds=1.5, sr=44100, normalize_input=True):
    """Run Apollo in overlapping chunks with Hann window (smoother boundaries, fewer artifacts).
    Apollo training rescales by max abs; we do the same. Returns [1, 1, T] on same device as audio."""
    chunk_samples = int(chunk_seconds * sr)
    hop_samples = int(overlap_hop_seconds * sr)
    B, C, total = audio.shape

    # Match Apollo training: rescale input by max absolute value
    if normalize_input:
        peak = audio.abs().max().clamp(min=1e-8)
        audio = audio / peak

    # Overlap-add with Hann window to avoid seam artifacts
    out_buf = torch.zeros_like(audio)
    weight_buf = torch.zeros(1, 1, total, device=audio.device, dtype=audio.dtype)
    window = torch.hann_window(chunk_samples, device=audio.device, dtype=audio.dtype).view(1, 1, -1)

    n_chunks = max(1, (total - chunk_samples + hop_samples) // hop_samples)
    start = 0
    idx = 0
    while start < total:
        end = min(start + chunk_samples, total)
        chunk = audio[:, :, start:end]
        if chunk.shape[2] < chunk_samples:
            pad = torch.zeros(B, C, chunk_samples - chunk.shape[2], device=audio.device, dtype=audio.dtype)
            chunk = torch.cat([chunk, pad], dim=2)
        with torch.no_grad():
            out = model(chunk)
        keep = end - start
        out_chunk = out[:, :, :keep]
        w = window[:, :, :keep]
        out_buf[:, :, start:start + keep] += out_chunk * w
        weight_buf[:, :, start:start + keep] += w
        idx += 1
        if idx <= 3 or idx % 20 == 0 or idx == n_chunks:
            print(f"  chunk {idx}/{n_chunks} ({100*start/total:.0f}%)", flush=True)
        start += hop_samples
        if start >= total:
            break

    weight_buf = weight_buf.clamp(min=1e-8)
    restored = out_buf / weight_buf

    # Rescale output to a safe peak (avoid clipping; match typical level)
    if normalize_input:
        restored = restored / (restored.abs().max().clamp(min=1e-8)) * 0.99

    return restored

In [15]:
# Input / output paths and restoration settings
# chunk_seconds: 3.0 matches Apollo training (best quality); reduce to 2.0 or 1.5 if GPU OOM
# overlap_hop_seconds: 1.5 = 50% overlap with 3s chunks (smooth seams); smaller = smoother but slower
input_track = os.path.join(app_dir, "asserts", "tuki_input.wav")
output_wav  = os.path.join(app_dir, "tuki_output.wav")
chunk_seconds = 3.0          # Match training segment length
overlap_hop_seconds = 1.5    # 50% overlap for smooth boundaries

audio = load_audio(input_track, target_sr=TARGET_SR)
duration_sec = audio.shape[2] / TARGET_SR
n_chunks_approx = max(1, (audio.shape[2] - int(chunk_seconds * TARGET_SR) + int(overlap_hop_seconds * TARGET_SR)) // int(overlap_hop_seconds * TARGET_SR))
print(f"Input: {duration_sec:.1f} s → ~{n_chunks_approx} chunks (restoration, not training — can take a few min on GPU, longer on CPU).")
restored = restore_in_chunks(
    model, audio,
    chunk_seconds=chunk_seconds,
    overlap_hop_seconds=overlap_hop_seconds,
    sr=TARGET_SR,
    normalize_input=True,   # match Apollo training (rescale by max abs)
)
save_audio(output_wav, restored, sr=TARGET_SR)
print(f"Restored audio saved to: {output_wav}")

Input: 194.4 s → ~128 chunks (restoration, not training — can take a few min on GPU, longer on CPU).
  chunk 1/128 (0%)
  chunk 2/128 (1%)
  chunk 3/128 (2%)
  chunk 20/128 (15%)
  chunk 40/128 (30%)
  chunk 60/128 (46%)
  chunk 80/128 (61%)
  chunk 100/128 (76%)
  chunk 120/128 (92%)
  chunk 128/128 (98%)
Restored audio saved to: /home/kagamirudo/CS614/Applications/tuki_output.wav
